In [1]:
!pip install -q transformers datasets accelerate

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
from transformers import DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments
from datasets import Dataset

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [4]:
model_name = "facebook/esm2_t6_8M_UR50D"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)

model = model.to(device)

config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/31.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/112 [00:00<?, ?it/s]

EsmForMaskedLM LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     |  | 
----------------------------+------------+--+-
esm.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
#This prints the full transformer architecture.
print(model)

EsmForMaskedLM(
  (esm): EsmModel(
    (embeddings): EsmEmbeddings(
      (word_embeddings): Embedding(33, 320, padding_idx=1)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): EsmEncoder(
      (layer): ModuleList(
        (0-5): 6 x EsmLayer(
          (attention): EsmAttention(
            (self): EsmSelfAttention(
              (query): Linear(in_features=320, out_features=320, bias=True)
              (key): Linear(in_features=320, out_features=320, bias=True)
              (value): Linear(in_features=320, out_features=320, bias=True)
              (rotary_embeddings): RotaryEmbedding()
            )
            (output): EsmSelfOutput(
              (dense): Linear(in_features=320, out_features=320, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (LayerNorm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
          )
          (intermediate): EsmIntermediate(
            (dense): Linear(in_features=320, out_feat

Embedding layer: Amino acids (33 tokens) are converted into 320-dimensional vectors.

Transformer encoder: The sequence passes through 6 transformer layers, each containing self-attention and feed-forward networks that learn relationships between residues.

LM head: The final layer predicts the probability of each amino acid at every position (masked language modeling).

Contact head: An auxiliary module predicts residue–residue contacts, useful for protein structure insights.

In [6]:
# This shows key architecture parameters
print(model.config)

EsmConfig {
  "add_cross_attention": false,
  "architectures": [
    "EsmForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.0,
  "classifier_dropout": null,
  "dtype": "float32",
  "emb_layer_norm_before": false,
  "esmfold_config": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 320,
  "initializer_range": 0.02,
  "intermediate_size": 1280,
  "is_decoder": false,
  "is_folding_model": false,
  "layer_norm_eps": 1e-05,
  "mask_token_id": 32,
  "max_position_embeddings": 1026,
  "model_type": "esm",
  "num_attention_heads": 20,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "position_embedding_type": "rotary",
  "tie_word_embeddings": true,
  "token_dropout": true,
  "transformers_version": "5.2.0",
  "use_cache": true,
  "vocab_list": null,
  "vocab_size": 33
}



Model size / representation

hidden_size = 320
Each amino-acid token is represented as a 320-dimensional embedding vector.

Transformer depth

num_hidden_layers = 6
The model has 6 transformer blocks, meaning the sequence is processed through 6 attention layers.

Attention structure

num_attention_heads = 20
Each layer has 20 self-attention heads that learn different residue–residue relationships.

Feed-forward network size

intermediate_size = 1280
Inside each transformer layer, the feed-forward network expands 320 → 1280 → 320, giving the model nonlinear capacity.

Sequence length limit

max_position_embeddings = 1026
The model can process protein sequences up to ~1026 residues.

Protein vocabulary

vocab_size = 33
The tokenizer supports 33 tokens (20 amino acids + special tokens like mask, pad, start, etc.).

In [7]:
# Count model parameters
num_params = sum(p.numel() for p in model.parameters())
print("Total parameters:", num_params)

Total parameters: 7512474


In [8]:
# Inspect transformer layers
for name, module in model.named_modules():
    if "layer" in name:
        print(name)

esm.encoder.layer
esm.encoder.layer.0
esm.encoder.layer.0.attention
esm.encoder.layer.0.attention.self
esm.encoder.layer.0.attention.self.query
esm.encoder.layer.0.attention.self.key
esm.encoder.layer.0.attention.self.value
esm.encoder.layer.0.attention.self.rotary_embeddings
esm.encoder.layer.0.attention.output
esm.encoder.layer.0.attention.output.dense
esm.encoder.layer.0.attention.output.dropout
esm.encoder.layer.0.attention.LayerNorm
esm.encoder.layer.0.intermediate
esm.encoder.layer.0.intermediate.dense
esm.encoder.layer.0.output
esm.encoder.layer.0.output.dense
esm.encoder.layer.0.output.dropout
esm.encoder.layer.0.LayerNorm
esm.encoder.layer.1
esm.encoder.layer.1.attention
esm.encoder.layer.1.attention.self
esm.encoder.layer.1.attention.self.query
esm.encoder.layer.1.attention.self.key
esm.encoder.layer.1.attention.self.value
esm.encoder.layer.1.attention.self.rotary_embeddings
esm.encoder.layer.1.attention.output
esm.encoder.layer.1.attention.output.dense
esm.encoder.layer.1.at

In [9]:
print(model.esm.encoder.layer[0])

EsmLayer(
  (attention): EsmAttention(
    (self): EsmSelfAttention(
      (query): Linear(in_features=320, out_features=320, bias=True)
      (key): Linear(in_features=320, out_features=320, bias=True)
      (value): Linear(in_features=320, out_features=320, bias=True)
      (rotary_embeddings): RotaryEmbedding()
    )
    (output): EsmSelfOutput(
      (dense): Linear(in_features=320, out_features=320, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (LayerNorm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
  )
  (intermediate): EsmIntermediate(
    (dense): Linear(in_features=320, out_features=1280, bias=True)
  )
  (output): EsmOutput(
    (dense): Linear(in_features=1280, out_features=320, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (LayerNorm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
)


In [10]:
seq = "MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQLR"

inputs = tokenizer(seq, return_tensors="pt")
inputs = {k:v.to(device) for k,v in inputs.items()}

outputs = model(**inputs)

# (batch_size, sequence_length, vocab_size)
print(outputs.logits.shape)

torch.Size([1, 37, 33])
